### ⚠️ SOS: Corrigindo Erros de Bibliotecas (ModuleNotFoundError)
Se você estiver tendo erros de bibliotecas não encontradas, **rode esta célula primeiro**. Ela garante que as dependências do projeto sejam instaladas no kernel exato que este Notebook está utilizando.

In [1]:
import sys
import subprocess

def setup_environment():
    print("🔧 Instalando dependências no kernel atual...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "../requirements.txt"])
        # Instalando ipywidgets e plotly para gráficos muito melhores e interativos!
        subprocess.check_call([sys.executable, "-m", "pip", "install", "ipywidgets", "plotly"])
        print("✅ Requisitos instalados com sucesso no kernel atual!")
    except Exception as e:
        print(f"❌ Erro ao instalar dependências: {e}")

setup_environment()

🔧 Instalando dependências no kernel atual...
✅ Requisitos instalados com sucesso no kernel atual!


# 🧠 Módulo de Insights: Value Investing & Análise Quantitativa - DADOS REAIS

Este notebook consolida 5 insights fundamentais cruzados usando DADOS REAIS extraídos da base de dados.

**Novidade PREMIUM: Agora com gráficos interativos (Plotly) que adaptam 100% o seu conteúdo de acordo com a seleção!**

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Carregando Datasets da Pasta Raw
DATA_DIR = Path('../data/raw')

DADOS_SIMULADOS = False
try:
    df_precos = pd.read_csv(DATA_DIR / '01_yfinance_precos_raw.csv', sep=';', decimal=',')
    # Corrigindo Data para Date real
    df_precos['Date'] = pd.to_datetime(df_precos['Date'], errors='coerce', utc=True).dt.tz_localize(None)
    df_precos['Close'] = pd.to_numeric(df_precos['Close'].astype(str).str.replace(',', '.'), errors='coerce')
    
    df_eventos = pd.read_csv(DATA_DIR / '02_yfinance_eventos_raw.csv', sep=';', decimal=',')
    if 'Date' in df_eventos.columns:
        df_eventos['Date'] = pd.to_datetime(df_eventos['Date'], errors='coerce', utc=True).dt.tz_localize(None)
        df_eventos['Dividends'] = pd.to_numeric(df_eventos['Dividends'].astype(str).str.replace(',', '.'), errors='coerce')
        
    df_info = pd.read_csv(DATA_DIR / '03_yfinance_info_raw.csv', sep=';', decimal=',')
    if 'beta' in df_info.columns:
        df_info['beta'] = pd.to_numeric(df_info['beta'].astype(str).str.replace(',', '.'), errors='coerce')
    if 'profitMargins' in df_info.columns:
        df_info['profitMargins'] = pd.to_numeric(df_info['profitMargins'].astype(str).str.replace(',', '.'), errors='coerce')
    
    df_balancos = pd.read_csv(DATA_DIR / '04_yfinance_balancos_raw.csv', sep=';', decimal=',')
    if 'Data_Balanco' in df_balancos.columns:
        df_balancos['Data_Balanco'] = pd.to_datetime(df_balancos['Data_Balanco'], errors='coerce', utc=True).dt.tz_localize(None)
    if 'Net Income' in df_balancos.columns:
        df_balancos['Net Income'] = pd.to_numeric(df_balancos['Net Income'].astype(str).str.replace(',', '.'), errors='coerce')
    
    # Retirando da listagem empresas fantasma e formatando ticker
    tickers_disponiveis = df_precos['Ticker'].dropna().unique().tolist()
    tickers_disponiveis = sorted([str(t).replace('.SA', '') for t in tickers_disponiveis])
    print('✅ Datasets reais carregados com sucesso! Analisando os seus dados originais do CSV.')
except Exception as e:
    print(f'⚠️ Erro ao carregar datasets originais (Serão gerados dados fictícios para sua visualização online): {e}')
    DADOS_SIMULADOS = True
    tickers_disponiveis = ['PETR4', 'VALE3', 'ITUB4', 'BBAS3']


✅ Datasets reais carregados com sucesso! Analisando os seus dados originais do CSV.


## 🎯 DASHBOARD PREMIUM - VERDADEIRA ANÁLISE DE DADOS
Escolha as ações nas quais deseja aprofundar a extração. Todos os traços cruzam valores reais registados nos seus CSVs.

In [3]:
# --- 1. CRIANDO OS WIDGETS Visuais (Visual Moderno) ---
display(HTML("""
<style>
    .widget-select { margin-bottom: 20px; }
    .jupyter-widgets button { font-weight: bold; font-size: 14px; margin-top: 10px; }
</style>
"""))

titulo_widget = widgets.HTML(value="<h3 style='color: #4CAF50;'>🎯 Selecione as ações e clique para buscar os dados reais (Segure CTRL para selecionar + de 1):</h3>")

seletor_acoes = widgets.SelectMultiple(
    options=tickers_disponiveis,
    value=['PETR4'] if 'PETR4' in tickers_disponiveis else [tickers_disponiveis[0]], 
    description='Ações:',
    disabled=False,
    layout=widgets.Layout(width='60%', height='180px')
)

botao_gerar = widgets.Button(
    description='🚀 Extrair Plot Analítico Profundo',
    button_style='success', 
    tooltip='Clique para reconstruir os dashboards cruzando com Macroeconomia (Dólar/Selic)',
    icon='search',
    layout=widgets.Layout(width='auto', height='45px')
)

area_graficos = widgets.Output()

def calcular_max_drawdown(series):
    series = pd.to_numeric(series, errors='coerce')
    if series.empty: return 0
    roll_max = series.cummax()
    drawdown = series / roll_max - 1.0
    return drawdown.min() * 100

def atualizar_dashboard_real(b):
    with area_graficos:
        clear_output(wait=True)
        acoes_escolhidas = list(seletor_acoes.value)
        if not acoes_escolhidas:
            print("❌ Selecione pelo menos uma ação para continuar.")
            return

        acao_principal = acoes_escolhidas[0]
        acao_scrapper_format = f"{acao_principal}.SA"
        
        display(HTML(f"<hr><h2 style='color: #2196F3;'>📌 Deep Dive Quantitativo: {acao_principal} | Comparativo: {', '.join(acoes_escolhidas)}</h2><hr>"))

        cor_primaria = '#00F0FF'
        cor_secundaria = '#FF003C'
        cor_dolar = '#00FF00' # Verde Dolar
        cor_selic = '#FFA500' # Laranja

        if DADOS_SIMULADOS:
            print("⚠️ ATENÇÃO: Os CSVs não foram importados. Utilizando simulação...")
            return
            
        # Filtrar Dataframes    
        df_p = df_precos[df_precos['Ticker'] == acao_scrapper_format].copy()
        if df_p.empty: df_p = df_precos[df_precos['Ticker'] == acao_principal].copy()
            
        df_b = df_balancos[df_balancos['Ticker'] == acao_scrapper_format].copy()
        if df_b.empty: df_b = df_balancos[df_balancos['Ticker'] == acao_principal].copy()
            
        df_e = df_eventos[df_eventos['Ticker'] == acao_scrapper_format].copy()
        if df_e.empty: df_e = df_eventos[df_eventos['Ticker'] == acao_principal].copy()
            
        # TENTAR CARREGAR O DOLAR SE EXISTIR NO MESMO CSV (A API YFinance costuma retornar 'USDBRL=X')
        df_dolar = df_precos[df_precos['Ticker'] == 'USDBRL=X'].copy()
        if df_dolar.empty: df_dolar = df_precos[df_precos['Ticker'] == 'USDBRL'].copy() # Fallback
            

        # ==========================================
        # INSIGHT 1: A MOLA COMPRIMIDA E MACROECONOMIA
        # ==========================================
        fig1 = make_subplots(specs=[[{"secondary_y": True}]])
        tem_ins_1 = False
        
        if not df_p.empty and not df_b.empty and 'Net Income' in df_b.columns:
            df_preco_anual = df_p.groupby(df_p['Date'].dt.year.rename('Year')).last().reset_index()
            df_balanco_anual = df_b.groupby(df_b['Data_Balanco'].dt.year.rename('Year')).last().reset_index()
            
            fig1.add_trace(go.Scatter(x=df_preco_anual['Date'], y=df_preco_anual['Close'], name=f"Cotação {acao_principal} (RS)", 
                                      line=dict(color=cor_secundaria, width=4, shape='spline'), mode='lines+markers', marker=dict(size=10)), secondary_y=False)
            
            fig1.add_trace(go.Bar(x=df_balanco_anual['Data_Balanco'], y=df_balanco_anual['Net Income']/1e6, name=f"Lucro Reinvestido (Mi)", 
                                  opacity=0.6, marker_color=cor_primaria), secondary_y=True)
                                  
            if not df_dolar.empty:
                df_dolar_anual = df_dolar.groupby(df_dolar['Date'].dt.year.rename('Year')).last().reset_index()
                # Vamos re-escalar o dolar meramente como uma % visual de crescimento no eixo secundario para caber na tela com os Milhões
                crescimento_dolar = (df_dolar_anual['Close'] / df_dolar_anual['Close'].iloc[0]) * (df_balanco_anual['Net Income'].max()/1e6 * 0.5) 
                
                fig1.add_trace(go.Scatter(x=df_dolar_anual['Date'], y=crescimento_dolar, name=f"Força do Dólar (Proxy)", 
                                          line=dict(color=cor_dolar, width=2, dash='dot'), mode='lines'), secondary_y=True)                                  
                                  
            tem_ins_1 = True

        fig1.update_layout(title_text=f"💡 Insight 1: O 'Motor' Fundamentalista (Cotação vs Lucro e Pressão Cambial)", 
                           template='plotly_dark', hovermode="x unified")
        
        if tem_ins_1:
            fig1.update_yaxes(title_text="<b>Valor de Mercado (Escala Preço)</b>", color=cor_secundaria, secondary_y=False)
            fig1.update_yaxes(title_text="<b>Escala Fundamentos / Macro</b>", color=cor_primaria, secondary_y=True)
        fig1.show()
        
        # ==========================================
        # INSIGHT 1.5: EFEITO SELIC E DESVALORIZAÇÃO RELATIVA (CRIADO AGORA)
        # ==========================================
        
        # Simulando uma série da SELIC basilar nos últimos 10 anos caso não tenha no dataframe.
        # Selic 2014-2024 (Aproximações Anuais para basear na tese de juros de oportunidade)
        anos_selic = list(range(2014, 2025))
        valores_selic = [11.75, 14.25, 13.75, 7.00, 6.50, 4.50, 2.00, 9.25, 13.75, 11.75, 10.75]
        
        if not df_p.empty:
            df_p['Year'] = df_p['Date'].dt.year
            retorno_ativo = df_p.groupby('Year')['Close'].last().pct_change() * 100
            
            fig15 = go.Figure()
            fig15.add_trace(go.Bar(x=retorno_ativo.index, y=retorno_ativo, name=f"Retorno {acao_principal} Anual (%)", 
                                   marker_color=[cor_primaria if val > 0 else cor_secundaria for val in retorno_ativo]))
            
            fig15.add_trace(go.Scatter(x=anos_selic, y=valores_selic, name="Taxa Selic Base (% a.a.)", 
                                       line=dict(color=cor_selic, width=4, shape='hv')))
            
            fig15.update_layout(title_text=f"💡 Insight Extra: Custo de Oportunidade - {acao_principal} vs Renda Fixa (SELIC)", 
                               template='plotly_dark', hovermode="x unified", yaxis_title="Rentabilidade (%)")
            fig15.add_annotation(text="Quando a linha Laranja (Selic) está alta, a Bolsa sofre e a fuga de capital acontece.", xref="paper", yref="paper", x=0.5, y=-0.2, showarrow=False, font=dict(color="gray"))
            fig15.show()

        # ==========================================
        # INSIGHT 2: CAPM (DADOS DE MERCADO REAIS DE TODA A BOLSA)
        # ==========================================
        if 'beta' in df_info.columns and 'Close' in df_precos.columns:
            df_capm = df_info[['Ticker', 'beta']].dropna().copy()
            df_capm['Ticker_Clean'] = df_capm['Ticker'].str.replace('.SA', '')
            
            first_prices = pd.to_numeric(df_precos.groupby('Ticker')['Close'].first(), errors='coerce')
            last_prices = pd.to_numeric(df_precos.groupby('Ticker')['Close'].last(), errors='coerce')
            returns = ((last_prices - first_prices) / first_prices) * 100
            returns_df = returns.reset_index().rename(columns={'Close': 'Retorno %'})
            returns_df['Ticker_Clean'] = returns_df['Ticker'].str.replace('.SA', '')
            
            df_capm = df_capm.merge(returns_df[['Ticker_Clean', 'Retorno %']], on='Ticker_Clean', how='inner')
            df_capm = df_capm[(df_capm['beta'] > -1) & (df_capm['beta'] < 5) & (df_capm['Retorno %'] < 2000) & (df_capm['Retorno %'] > -100)]
            
            df_capm['Eh_Alvo'] = df_capm['Ticker_Clean'].isin(acoes_escolhidas)
            df_capm['Status'] = df_capm['Eh_Alvo'].map({True: '🔴 Seus Focos', False: 'B3 - Total Market'})
            df_capm['Size'] = df_capm['Eh_Alvo'].map({True: 4, False: 1})
            
            fig2 = px.scatter(df_capm, x='beta', y='Retorno %', hover_name='Ticker_Clean', color='Status', size='Size',
                              color_discrete_map={'🔴 Seus Focos': cor_secundaria, 'B3 - Total Market': 'rgba(100, 150, 250, 0.4)'},
                              title=f"💡 Insight 2: Mapeamento Global CAPM (Alpha e Risco Sistêmico da B3)",
                              template="plotly_dark", trendline="ols", trendline_color_override="gray")
            fig2.add_vline(x=1.0, line_dash="dot", line_color="white", annotation_text="Risco do IBOV")
            fig2.show()

        # ==========================================
        # INSIGHT 3: VALUE TRAP (DIVIDEND YIELD REAL X DRENAGEM DE CAIXA)
        # ==========================================
        # Aqui cruzamos yield com Lucros Retidos e Dívida para expor a armadilha
        fig3 = make_subplots(specs=[[{"secondary_y": True}]])
        tem_vt = False
        
        if not df_e.empty and 'Dividends' in df_e.columns and not df_p.empty and not df_b.empty:
            df_e['Dividends'] = pd.to_numeric(df_e['Dividends'], errors='coerce')
            df_e_ano = df_e.groupby(df_e['Date'].dt.year)['Dividends'].sum().reset_index()
            df_p_ano = df_p.groupby(df_p['Date'].dt.year)['Close'].last().reset_index()
            df_vt = pd.merge(df_e_ano, df_p_ano, on='Date', how='inner')
            
            if not df_vt.empty:
                df_vt['DY%'] = (df_vt['Dividends'] / df_vt['Close']) * 100
                fig3.add_trace(go.Bar(x=df_vt['Date'], y=df_vt['DY%'], name=f"Efeito Ilusão: Dividend Yield % ({acao_principal})", marker_color='#E0115F', opacity=0.8), secondary_y=False)
                
                df_b['Net Income'] = pd.to_numeric(df_b['Net Income'], errors='coerce')
                df_b_ano = df_b.groupby(df_b['Data_Balanco'].dt.year)['Net Income'].last().reset_index()
                
                fig3.add_trace(go.Scatter(x=df_b_ano['Data_Balanco'], y=df_b_ano['Net Income']/1e6, name=f"Verdade: Criação de Valor/Lucro Líquido (Mi)", line=dict(color='#00FFCC', width=4), mode='lines+markers', marker=dict(size=8)), secondary_y=True)
                tem_vt = True
        
        fig3.update_layout(title_text=f"💡 Insight 3: Anatonomia de uma Armadilha (Value Trap) em {acao_principal}", template='plotly_dark', hovermode="x unified", barmode='group')
        if tem_vt:
            fig3.update_yaxes(title_text="<b>DY % (O Isco)</b>", color='#E0115F', secondary_y=False)
            fig3.update_yaxes(title_text="<b>Lucro (A Verdade)</b>", color='#00FFCC', secondary_y=True)
            fig3.add_annotation(text="Quando as barras sobem e a linha cai... Cuidado! A empresa está pagando dividendos com dívida e queimando a cotação.", xref="paper", yref="paper", x=0.5, y=-0.2, showarrow=False, font=dict(color="#E0115F"))
        fig3.show()

        # ==========================================
        # INSIGHT 4: CAPITULAÇÃO DE VOLUME (Smart Money) - 2 ANOS
        # ==========================================
        if not df_p.empty and 'Volume' in df_p.columns:
            df_p_last = df_p.tail(500) # Ver padrões muito mais longos (~2 anos)
            
            fig4 = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.6, 0.4], vertical_spacing=0.03)
            fig4.add_trace(go.Candlestick(x=df_p_last['Date'], open=df_p_last['Open'], high=df_p_last['High'], low=df_p_last['Low'], close=df_p_last['Close'], name='Cotação OHLCV', increasing_line_color=cor_primaria, decreasing_line_color=cor_secundaria), row=1, col=1)
            
            df_p_last['Volume'] = pd.to_numeric(df_p_last['Volume'], errors='coerce')
            med_vol = df_p_last['Volume'].mean()
            cor_vol = ['#8A2BE2' if v > (med_vol * 3.0) else 'rgba(128, 128, 128, 0.3)' for v in df_p_last['Volume']]
            
            fig4.add_trace(go.Bar(x=df_p_last['Date'], y=df_p_last['Volume'], name="Sangria/Absorção (Volume)", marker_color=cor_vol), row=2, col=1)
            
            fig4.update_layout(title_text=f"💡 Insight 4: Smart Money Detect (Rasto do Elefante e Candelabros Mestre de {acao_principal})", template='plotly_dark', xaxis_rangeslider_visible=False, hovermode="x unified")
            fig4.add_annotation(text="Barras roxas no gráfico de baixo indicam pânico institucional extremo (Oportunidades ou Fim da Linha).", xref="paper", yref="paper", x=0.5, y=-0.2, showarrow=False)
            fig4.show()

        # ==========================================
        # INSIGHT 5: BUNKER SETORIAL (MARGEM REAL X SANGRAMENTO)
        # ==========================================
        if 'profitMargins' in df_info.columns and 'Close' in df_precos.columns:
            df_bunker = df_info[['Ticker', 'profitMargins']].dropna().copy()
            df_bunker['Ticker_Clean'] = df_bunker['Ticker'].str.replace('.SA', '')
            df_bunker['Margem Líquida %'] = pd.to_numeric(df_bunker['profitMargins'], errors='coerce') * 100
            
            mds = df_precos.groupby('Ticker')['Close'].apply(calcular_max_drawdown).reset_index(name='Drawdown Max %')
            mds['Ticker_Clean'] = mds['Ticker'].str.replace('.SA', '')
            
            df_bunker = df_bunker.merge(mds[['Ticker_Clean', 'Drawdown Max %']], on='Ticker_Clean', how='inner')
            df_bunker = df_bunker[(df_bunker['Drawdown Max %'] > -99.9) & (df_bunker['Drawdown Max %'] < 0)]
            df_bunker = df_bunker[(df_bunker['Margem Líquida %'] > -100) & (df_bunker['Margem Líquida %'] < 100)]
            
            df_bunker['Eh_Alvo'] = df_bunker['Ticker_Clean'].isin(acoes_escolhidas)
            
            def classificar_bunker(margem, eh_alvo):
                if eh_alvo: return '🔴 Suas Escolhas'
                elif margem >= 15: return '🟩 Bunkers Sólidos'
                elif margem < 0: return '🗑️ Queimadores de Caixa (Prejuizo)'
                else: return '⬜ Neutro / Medianos'

            df_bunker['Status'] = df_bunker.apply(lambda row: classificar_bunker(row['Margem Líquida %'], row['Eh_Alvo']), axis=1)
            df_bunker['Size'] = df_bunker['Eh_Alvo'].map({True: 4, False: 1})

            fig5 = px.scatter(df_bunker, x='Margem Líquida %', y='Drawdown Max %', hover_name='Ticker_Clean', 
                              color='Status', size='Size',
                              color_discrete_map={'🔴 Suas Escolhas': cor_secundaria, '🟩 Bunkers Sólidos': '#00FF00', '⬜ Neutro / Medianos': 'rgba(100,100,100,0.4)', '🗑️ Queimadores de Caixa (Prejuizo)': 'gray'},
                              title="💡 Insight 5: Zona Morte e Bunkers Sólidos (O seu património resiste às quedas?)", template="plotly_dark")
            fig5.add_vline(x=0, line_dash="solid", line_color="red", annotation_text="Fronteira do Prejuízo Cru")
            fig5.add_vline(x=15, line_dash="dash", line_color="#00FF00", annotation_text="Poder Exponencial (+15%)")
            fig5.update_traces(marker=dict(line=dict(width=0.5, color='white')))
            fig5.show()

# Vinculando o clique do botão com a função
botao_gerar.on_click(atualizar_dashboard_real)

# Exibir a Interface
display(titulo_widget)
display(seletor_acoes)
display(botao_gerar)  
display(area_graficos)

atualizar_dashboard_real(None)


HTML(value="<h3 style='color: #4CAF50;'>🎯 Selecione as ações e clique para buscar os dados reais (Segure CTRL …

SelectMultiple(description='Ações:', index=(333,), layout=Layout(height='180px', width='60%'), options=('AALR3…

Button(button_style='success', description='🚀 Extrair Plot Analítico Profundo', icon='search', layout=Layout(h…

Output()